# K26 - Heated Wall

Part of the Dissertation of Matthias Rieckmann, Section 7.3  
Interface at 90°.  
Equal fluid densities => simplified setting  
Also no Heat capacity => infinitely fast heat conduction  

In [ ]:
#r "BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

## Setup Workflowmanagement, Batchprocessor and Database

In [ ]:
ExecutionQueues

In [ ]:
static var myBatch = BoSSSshell.GetDefaultQueue();

In [ ]:
BoSSSshell.WorkflowMgm.Init($"3PhaseDemo");
BoSSSshell.WorkflowMgm.Sessions

In [ ]:
static var myDb = BoSSSshell.WorkflowMgm.DefaultDatabase;

In [ ]:
myDb.Path

In [ ]:
BoSSSshell.WorkflowMgm.SetNameBasedSessionJobControlCorrelation();

## Setup Simulationcontrols

In [ ]:
using BoSSS.Application.XNSFE_Solver;

In [ ]:
int[] hRes = {1};//{1, 2, 3, 4};
int[] pDeg = {2};//{ 1, 2, 3, 4};
double[] Q = {0.2};

In [ ]:
List<XNSFE_Control> Controls = new List<XNSFE_Control>();
foreach(int h in hRes){
    foreach(int p in pDeg){
        foreach(double q in Q){
             foreach(bool threephase in new[] {true, false}){

        var ctrl = new XNSFE_Control();

        ctrl.DbPath = null;
        ctrl.SetDatabase(myDb);
        ctrl.SessionName = (threephase ? "3" : "2")+"PhaseDemo";
        ctrl.ProjectName = $"3PhaseDemo";
        ctrl.savetodb = true;

        ctrl.SetDGdegree(p);

        #region grid
        double L = 1.0;
        int kelemR = h;
        string[] Bndy = new string[] {  "Inner",
                                    "NavierSlip_linear_ConstantHeatFlux_right",
                                    "pressure_outlet_ZeroGradient_top",
                                    "freeslip_ZeroGradient_left",
                                    "pressure_outlet_ZeroGradient_bottom"};

        ctrl.GridFunc = delegate () {
            double[] Xnodes = GenericBlas.Linspace(-5*L, threephase ? L : 0, (threephase ? 6 : 5) * kelemR + 1);
            double[] Ynodes = GenericBlas.Linspace(0, 5*L, 5 * kelemR + 1);
            var grd = Grid2D.Cartesian2DGrid(Xnodes, Ynodes);

            for (byte i = 1; i < Bndy.Count(); i++) {
                grd.EdgeTagNames.Add(i, Bndy[i]);
            }

            grd.DefineEdgeTags(delegate (double[] X) {
                byte et = 0;
                if (Math.Abs(X[0] - Xnodes.Last()) < 1e-8)
                    return 1;
                if (Math.Abs(X[0] - Xnodes.First()) < 1e-8)
                    return 3;
                if (Math.Abs(X[1] - Ynodes.Last()) < 1e-8)
                    return 2;
                if (Math.Abs(X[1] - Ynodes.First()) < 1e-8)
                    return 4;
                return et;
            });

            return grd;
        };
        #endregion

        #region material
        ctrl.PhysicalParameters = new BoSSS.Solution.XNSECommon.PhysicalParameters() {
            rho_A = 1.0, // 958.0
            rho_B = 1.0, // 0.59,

            mu_A = 1, //2.82 * 1e-4,
            mu_B = 0.001, //1.23 * 1e-6,

            theta_e = Math.PI / 180.0 * 90.0,
            Sigma = 1.0,
            betaS_A = 1000, // sliplength is mu/beta
            betaS_B = 1000,
        };

        ctrl.ThermalParameters = new BoSSS.Solution.XheatCommon.ThermalParameters() {
            rho_A = 1.0, // 958.0
            rho_B = 1.0, //0.59,
            rho_C = 1.0,

            k_A = 1.0, // 0.6
            k_B = 1.0, // 0.026,
            k_C = 1.0,

            c_A = 0.0,
            c_B = 0.0,
            c_C = 0.0,

            hVap = 1,//2.257 * 1e6,
            T_sat = 0.0 // 373.0
        };

        ctrl.PhysicalParameters.IncludeConvection = true;
        ctrl.ThermalParameters.IncludeConvection = true;
        ctrl.PhysicalParameters.Material = false;
        #endregion

        #region Initial Condition - Exact Solution

        // solution for massflux and velocity at level set
        double y0 = 2.2 * L;

        // inital values
        double g = 4;

        //double a = Math.Sqrt(ctrl.PhysicalParameters.Sigma / (ctrl.PhysicalParameters.rho_A * g));
        //double h = Math.Sqrt(2*a*a*(1-Math.Sin(ctrl.PhysicalParameters.theta_e)));

        ctrl.AddInitialValue("Phi", $"X => -{y0} + X[1]", false);

        if (threephase) {
            ctrl.UseImmersedBoundary = true;
            ctrl.AdvancedDiscretizationOptions.DoubleCutSpecialQuadrature = true;
            ctrl.AdvancedDiscretizationOptions.IBM_BoundaryType = BoSSS.Solution.XNSECommon.IBM_BoundaryType.NavierSlip;
            ctrl.AdvancedDiscretizationOptions.GNBC_Localization = BoSSS.Solution.XNSECommon.NavierSlip_Localization.Everywhere;
            ctrl.AddInitialValue("Phi2", $"(X, t) => X[0]", true);
            ctrl.AddInitialValue("Temperature#C", $"(X, t) => {ctrl.ThermalParameters.T_sat}", true);
            ctrl.AddInitialValue("VelocityX@Phi2", $"(X, t) => 0", true);
            ctrl.AddInitialValue("VelocityY@Phi2", $"(X, t) => 0", true);
            ctrl.HeatSourceIBM = new Dictionary<string, string>();
            ctrl.HeatSourceIBM["AC"] = $"(X,t) => {q}";
            ctrl.AddBoundaryValue(Bndy[1]);

        } else {
            ctrl.AddBoundaryValue(Bndy[1], "HeatFluxX#A", $"X => {q}", false);
        }

        ctrl.AddInitialValue("Temperature#A", $"(X, t) => {ctrl.ThermalParameters.T_sat}", true);
        ctrl.AddInitialValue("Temperature#B", $"(X, t) => {ctrl.ThermalParameters.T_sat}", true);
        ctrl.AddInitialValue("GravityY#A", $"(X, t) => -{g}", true);



        #endregion

        #region Boundary Conditions

        ctrl.AddBoundaryValue(Bndy[3]);
        ctrl.AddBoundaryValue(Bndy[2]);
        ctrl.AddBoundaryValue(Bndy[4], "Pressure#A", $"(X, t) => {y0} * {ctrl.PhysicalParameters.rho_A} * {g}", true);

        #endregion

        #region AMR

        int level = 5;
            ctrl.AdaptiveMeshRefinement = level > 0;
            ctrl.activeAMRlevelIndicators.Add(new BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater.AMRonNarrowband() { maxRefinementLevel = level });
            if (threephase) {
                ctrl.activeAMRlevelIndicators.Add(new BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater.AMRatInnerContactLine() { maxRefinementLevel = 2 * level });
            } else {
                ctrl.activeAMRlevelIndicators.Add(new BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater.AMRatContactLine() { edgeTags = new byte[] { 1 }, maxRefinementLevel = 2 * level });
            }

            ctrl.AMR_startUpSweeps = 2 * level;

        #endregion

        #region Timestepping

        ctrl.AdvancedDiscretizationOptions.SST_isotropicMode = BoSSS.Solution.LevelSetTools.SurfaceStressTensor_IsotropicMode.LaplaceBeltrami_ContactLine;
        ctrl.Option_LevelSetEvolution = BoSSS.Solution.LevelSetTools.LevelSetEvolution.None;
        ctrl.Timestepper_LevelSetHandling = BoSSS.Solution.XdgTimestepping.LevelSetHandling.None;

        ctrl.NonLinearSolver.SolverCode = NonLinearSolverCode.Newton;
        ctrl.NonLinearSolver.Globalization = BoSSS.Solution.AdvancedSolvers.Newton.GlobalizationOption.Dogleg;
        ctrl.NonLinearSolver.ConvergenceCriterion = 1e-8;
        ctrl.NonLinearSolver.MaxSolverIterations = 10;

        ctrl.SkipSolveAndEvaluateResidual = false;

        ctrl.TimeSteppingScheme = BoSSS.Solution.XdgTimestepping.TimeSteppingScheme.ImplicitEuler;
        ctrl.TimesteppingMode = BoSSS.Solution.Control.AppControl._TimesteppingMode.Steady;
        //ctrl.dtFixed = 0.01;
        //ctrl.Endtime = 15.0;
        //ctrl.NoOfTimesteps = (int)(ctrl.Endtime / ctrl.dtFixed);

        #endregion
        //if (threephase) {
        //    ctrl.PostprocessingModules.Add(new BoSSS.Application.XNSFE_Solver.PhysicalBasedTestcases.MovingContactLineZwoLsLogging() { LogPeriod = 1 });
        //} else {
        //    ctrl.PostprocessingModules.Add(new BoSSS.Application.XNSFE_Solver.PhysicalBasedTestcases.MassfluxLogging() { LogPeriod = 1 });
        //}
            
        Controls.Add(ctrl);
             }
        }
    }
}

In [ ]:
int N = Controls.Count;

## Start simulations on Batch processor

In [ ]:
Controls.RunBatch(myBatch, false);

In [ ]:
BoSSSshell.WorkflowMgm.BlockUntilAllJobsTerminate(3600);

In [ ]:
int count = BoSSSshell.wmg.Sessions.Count();
int success = BoSSSshell.wmg.Sessions.Where(s => s.SuccessfulTermination).Count();

if(count != Controls.Count() || count != success){
    throw new ApplicationException("Not all simulations calculated or finished successful!");
}

## Postprocessing

In [ ]:
var sessions = myDb.Sessions;

In [ ]:
//sessions.ForEach(s => s.Export().To(Path.GetFullPath(Path.Combine(".", s.ProjectName, s.Name))).WithShadowFields().WithSupersampling(3).Do());

In [ ]:
using MathNet.Numerics.Interpolation;
using BoSSS.Foundation.Quadrature;
using System.IO;
using System.Text;

In [ ]:
 // Helper function to create splines on edges
static public CubicSpline[] SplineOnInterface(int levelset, XDGField field, int d, double offset = 0.0) {
    var LsTrk = field.Basis.Tracker;
    var grd = LsTrk.GridDat;

    List<double>[] nodes = LsTrk.TotalNoOfSpecies.ForLoop(i => new List<double>());
    List<double>[] values = LsTrk.TotalNoOfSpecies.ForLoop(i => new List<double>());

    int momentFittingOrder = field.Basis.Degree * 2;
    var SchemeHelper = LsTrk.GetXDGSpaceMetrics(LsTrk.SpeciesIdS,  momentFittingOrder, 1).XQuadSchemeHelper;
    CellQuadratureScheme cqs = SchemeHelper.GetLevelSetquadScheme(levelset, LsTrk.Regions.GetCutCellMask());

    
    CellQuadrature.GetQuadrature(new int[] { 1 }, grd,
        cqs.Compile(LsTrk.GridDat, momentFittingOrder),
        delegate (int i0, int Length, QuadRule QR, MultidimensionalArray EvalResult) {

            MultidimensionalArray Res = MultidimensionalArray.Create(Length, QR.NoOfNodes, LsTrk.TotalNoOfSpecies);
            MultidimensionalArray GlobalNodes = MultidimensionalArray.Create(Length, QR.NoOfNodes, grd.SpatialDimension);
            grd.TransformLocal2Global(QR.Nodes, i0, Length, GlobalNodes, 0);

            for(int j = 0; j < LsTrk.TotalNoOfSpecies; j++){
                field.GetSpeciesShadowField(LsTrk.SpeciesNames.ElementAt(j)).Evaluate(i0, Length, QR.Nodes, Res.ExtractSubArrayShallow(-1,-1, j));
                for (int i = 0; i < Length; i++) {
                    int K = QR.NoOfNodes;
                    for (int k = 0; k < K; k++) {
                        nodes[j].Add(GlobalNodes[i, k, d]);
                        values[j].Add(Res[i, k, j]);
                    }
                }
            }
            
            

        }, delegate (int i0, int Length, MultidimensionalArray ResultsOfIntegration) {
        }).Execute();


    return LsTrk.TotalNoOfSpecies.ForLoop(i => CubicSpline.InterpolateAkima(nodes[i].ToArray(), values[i].ToArray()));
}

static public CubicSpline[] SplineOnInterfaceGradient(int levelset, XDGField field, int dd, int d, double offset = 0.0) {
    var LsTrk = field.Basis.Tracker;
    var grd = LsTrk.GridDat;

    List<double>[] nodes = LsTrk.TotalNoOfSpecies.ForLoop(i => new List<double>());
    List<double>[] values = LsTrk.TotalNoOfSpecies.ForLoop(i => new List<double>());

    int momentFittingOrder = field.Basis.Degree * 2;
    var SchemeHelper = LsTrk.GetXDGSpaceMetrics(LsTrk.SpeciesIdS,  momentFittingOrder, 1).XQuadSchemeHelper;
    CellQuadratureScheme cqs = SchemeHelper.GetLevelSetquadScheme(levelset, LsTrk.Regions.GetCutCellMask());

    
    CellQuadrature.GetQuadrature(new int[] { 1 }, grd,
        cqs.Compile(LsTrk.GridDat, momentFittingOrder),
        delegate (int i0, int Length, QuadRule QR, MultidimensionalArray EvalResult) {

            MultidimensionalArray Res = MultidimensionalArray.Create(Length, QR.NoOfNodes, grd.SpatialDimension, LsTrk.TotalNoOfSpecies);
            MultidimensionalArray GlobalNodes = MultidimensionalArray.Create(Length, QR.NoOfNodes, grd.SpatialDimension);
            grd.TransformLocal2Global(QR.Nodes, i0, Length, GlobalNodes, 0);

            for(int j = 0; j < LsTrk.TotalNoOfSpecies; j++){
                field.GetSpeciesShadowField(LsTrk.SpeciesNames.ElementAt(j)).EvaluateGradient(i0, Length, QR.Nodes, Res.ExtractSubArrayShallow(-1,-1, -1, j));
                for (int i = 0; i < Length; i++) {
                    int K = QR.NoOfNodes;
                    for (int k = 0; k < K; k++) {
                        nodes[j].Add(GlobalNodes[i, k, d]);
                        values[j].Add(Res[i, k, dd, j]);
                    }
                }
            }
            
            

        }, delegate (int i0, int Length, MultidimensionalArray ResultsOfIntegration) {
        }).Execute();


    return LsTrk.TotalNoOfSpecies.ForLoop(i => CubicSpline.InterpolateAkima(nodes[i].ToArray(), values[i].ToArray()));
}

static public CubicSpline[] SplineOnEdge(EdgeMask em, XDGField field, int d) {
    var LsTrk = field.Basis.Tracker;
    var grd = LsTrk.GridDat;

    List<double>[] nodes = LsTrk.TotalNoOfSpecies.ForLoop(i => new List<double>());
    List<double>[] values = LsTrk.TotalNoOfSpecies.ForLoop(i => new List<double>());

    int momentFittingOrder = field.Basis.Degree * 2;

    EdgeQuadrature.GetQuadrature(new int[] { 1 }, grd,
        (new EdgeQuadratureScheme(true, em)).Compile(grd, momentFittingOrder),
        delegate (int i0, int Length, QuadRule QR, MultidimensionalArray EvalResult) {

            MultidimensionalArray DummyIN = MultidimensionalArray.Create(Length, QR.NoOfNodes, LsTrk.TotalNoOfSpecies);
            MultidimensionalArray DummyOT = MultidimensionalArray.Create(Length, QR.NoOfNodes);

            MultidimensionalArray GlobalNodes = MultidimensionalArray.Create(Length, QR.NoOfNodes, grd.SpatialDimension);

            for(int j = 0; j < LsTrk.TotalNoOfSpecies; j++){
                field.GetSpeciesShadowField(LsTrk.SpeciesNames.ElementAt(j)).EvaluateEdge(i0, Length, QR.Nodes, DummyIN.ExtractSubArrayShallow(-1,-1, j), DummyOT, null, null, null, null, 0, 0.0);
            }

            
            for (int i = 0; i < Length; i++) {
                int iTrafo = ((GridData)grd).Edges.Edge2CellTrafoIndex[i0 + i, 0];
                NodeSet volNodeSet = QR.Nodes.GetVolumeNodeSet(grd, iTrafo, false);
                int jCell = ((GridData)grd).Edges.CellIndices[i0 + i, 0];
                grd.TransformLocal2Global(volNodeSet, jCell, 1, GlobalNodes, i);
                int K = QR.NoOfNodes;
                for (int k = 0; k < K; k++) {
                    for(int j = 0; j < LsTrk.TotalNoOfSpecies; j++){
                        nodes[j].Add(GlobalNodes[i, k, d]);
                        values[j].Add(DummyIN[i, k, j]);
                    }
                }
            }

        }, delegate (int i0, int Length, MultidimensionalArray ResultsOfIntegration) {
        }).Execute();


    return LsTrk.TotalNoOfSpecies.ForLoop(i => CubicSpline.InterpolateAkima(nodes[i].ToArray(), values[i].ToArray()));
}


static public CubicSpline[] SplineOnEdgeGradient(EdgeMask em, XDGField field, int dd, int d) {
    var LsTrk = field.Basis.Tracker;
    var grd = LsTrk.GridDat;

    List<double>[] nodes = LsTrk.TotalNoOfSpecies.ForLoop(i => new List<double>());
    List<double>[] values = LsTrk.TotalNoOfSpecies.ForLoop(i => new List<double>());

    int momentFittingOrder = field.Basis.Degree * 2;

    EdgeQuadrature.GetQuadrature(new int[] { 1 }, grd,
        (new EdgeQuadratureScheme(true, em)).Compile(grd, momentFittingOrder),
        delegate (int i0, int Length, QuadRule QR, MultidimensionalArray EvalResult) {

            MultidimensionalArray DummyIN = MultidimensionalArray.Create(Length, QR.NoOfNodes, grd.SpatialDimension, LsTrk.TotalNoOfSpecies);

            MultidimensionalArray GlobalNodes = MultidimensionalArray.Create(Length, QR.NoOfNodes, grd.SpatialDimension);  
            
            for (int i = 0; i < Length; i++) {
                int iTrafo = ((GridData)grd).Edges.Edge2CellTrafoIndex[i0 + i, 0];
                NodeSet volNodeSet = QR.Nodes.GetVolumeNodeSet(grd, iTrafo, false);
                int jCell = ((GridData)grd).Edges.CellIndices[i0 + i, 0];
                grd.TransformLocal2Global(volNodeSet, jCell, 1, GlobalNodes, i);   
                int K = QR.NoOfNodes;
                for(int j = 0; j < LsTrk.TotalNoOfSpecies; j++){
                    field.GetSpeciesShadowField(LsTrk.SpeciesNames.ElementAt(j)).EvaluateGradient(jCell, 1, volNodeSet, DummyIN.ExtractSubArrayShallow(-1,-1, -1, j), i);
                }
                for (int k = 0; k < K; k++) {
                    for(int j = 0; j < LsTrk.TotalNoOfSpecies; j++){
                        nodes[j].Add(GlobalNodes[i, k, d]);
                        values[j].Add(DummyIN[i, k, dd, j]);
                    }
                }
            }

        }, delegate (int i0, int Length, MultidimensionalArray ResultsOfIntegration) {
        }).Execute();


    return LsTrk.TotalNoOfSpecies.ForLoop(i => CubicSpline.InterpolateAkima(nodes[i].ToArray(), values[i].ToArray()));
}

void ExportData(string filename, double[] x, double[] y){
    using(var stw = new StringWriter()) {   

        int n = x.Length;
        for(int i = 0; i < n; i++){
            stw.WriteLine($"{x[i].ToString()} {y[i].ToString()}");
        }
        
        Directory.CreateDirectory(BoSSSshell.wmg.CurrentProject);
        File.WriteAllText(Path.Combine(".",BoSSSshell.wmg.CurrentProject, filename), stw.ToString());
    }
    
}

Extract temperature and temperature gradient profiles along the fluid solid and the fluid-fluid interface

In [ ]:
var sess = sessions.Where(s => s.Name == "3PhaseDemo").Single();
var ts = sess.Timesteps.Last();
var fields = ts.Fields;
var temp = (XDGField)fields.Where(f => f.Identification.Contains("Temperature")).Single();
var LsTrk = temp.Basis.Tracker;
for(int j = 0; j < LsTrk.TotalNoOfSpecies; j++){
    LsTrk.SpeciesNames.ElementAt(j).Display();
}
var P3LS1tempSplines = SplineOnInterface(1, temp, 1);
var P3LS1tempGradXSplines = SplineOnInterfaceGradient(1, temp, 0, 1);
var P3LS0tempSplines = SplineOnInterface(0, temp, 0);
var P3LS0tempGradYSplines = SplineOnInterfaceGradient(0, temp, 1, 0);

In [ ]:
var sess = sessions.Where(s => s.Name == "2PhaseDemo").Single();
var ts = sess.Timesteps.Last();
var fields = ts.Fields;
var temp = (XDGField)fields.Where(f => f.Identification.Contains("Temperature")).Single();
var LsTrk = temp.Basis.Tracker;
for(int j = 0; j < LsTrk.TotalNoOfSpecies; j++){
    LsTrk.SpeciesNames.ElementAt(j).Display();
}
var em = new EdgeMask(LsTrk.GridDat, X => Math.Abs(X[0]) < 1e-12);
var P2LS1tempSplines = SplineOnEdge(em, temp, 1);
var P2LS1tempGradXSplines = SplineOnEdgeGradient(em, temp, 0, 1);
var P2LS0tempSplines = SplineOnInterface(0, temp, 0);
var P2LS0tempGradYSplines = SplineOnInterfaceGradient(0, temp, 1, 0);

Visually compare the temperature profile along the fluid solid interface.  
In the true three-phase situation the temperature is displayed from the fluid side ($A/B$) and the solid side ($C$), they should coincide (no temperature "slip").  
In the two phase case only the fluid sides are computed.

In [ ]:
int N = 1000;
var x = GenericBlas.Linspace(0, 2.2, N).Cat(GenericBlas.Linspace(2.2, 5, N));

var tab3= x.Select(xx => xx < 2.2 ? P3LS1tempSplines[0].Interpolate(xx) : P3LS1tempSplines[2].Interpolate(xx)).ToArray();
var tc3 = x.Select(xx => P3LS1tempSplines[1].Interpolate(xx)).ToArray();

var tab2 = x.Select(xx => xx < 2.2 ? P2LS1tempSplines[0].Interpolate(xx) : P2LS1tempSplines[1].Interpolate(xx)).ToArray();

Gnuplot plt = new Gnuplot();

plt.PlotXY(x, tab3, title : "Temperature A / B - 3 Phases", format: new PlotFormat("-r"));
plt.PlotXY(x, tc3, title : "Temperature C - 3 Phases", format: new PlotFormat("--k"));

plt.PlotXY(x, tab2, title : "Temperature A / B - 2 Phases", format: new PlotFormat("--r"));

plt.Cmd("set grid");
plt.PlotNow().Display();

//ExportData($"Temperature3_AB_1.txt",x, tab3);
//ExportData($"Temperature3_C_1.txt",x, tc3);

//ExportData($"Temperature2_AB_1.txt",x, tab2);


The same consideration but for heat fluxes in interface normal direction.

In [ ]:
var txab3 = x.Select(xx => xx < 2.2 ? -P3LS1tempGradXSplines[0].Interpolate(xx) : -P3LS1tempGradXSplines[2].Interpolate(xx)).ToArray();
var txc3 = x.Select(xx => -1 * P3LS1tempGradXSplines[1].Interpolate(xx)).ToArray();
var tg3 = x.Select(xx => (xx < 2.2 ? -P3LS1tempGradXSplines[0].Interpolate(xx) : -P3LS1tempGradXSplines[2].Interpolate(xx)) + 1 * P3LS1tempGradXSplines[1].Interpolate(xx)).ToArray();

var txab2 = x.Select(xx => xx < 2.2 ? -P2LS1tempGradXSplines[0].Interpolate(xx) : -P2LS1tempGradXSplines[1].Interpolate(xx)).ToArray();

Gnuplot plt = new Gnuplot();

plt.PlotXY(x, txab3.Select(xx => Math.Abs(xx)), title : "HeatFlux A / B - 3 Phases", format: new PlotFormat("-r"));
plt.PlotXY(x, txc3.Select(xx => Math.Abs(xx)), title : "HeatFlux C - 3 Phases", format: new PlotFormat("--k"));
plt.PlotXY(x, tg3.Select(xx => Math.Abs(xx)), title : "HeatFlux Total - 3 Phases", format: new PlotFormat("--b"));

plt.PlotXY(x, txab2.Select(xx => Math.Abs(xx)), title : "HeatFlux A / B - 2 Phases", format: new PlotFormat("--r"));


plt.Cmd("set grid");
plt.PlotNow().Display();

//ExportData($"HeatFlux3_AB_1.txt",x, txab3);
//ExportData($"HeatFlux3_C_1.txt",x, txc3);

//ExportData($"HeatFlux2_AB_1.txt",x, txab2);

Visually compare the temperature profile along the fluid-fluid interface.  

In [ ]:
int N = 1000;
var x = GenericBlas.Linspace(-1, 0, N);

var ta3 = x.Select(xx => P3LS0tempSplines[0].Interpolate(xx)).ToArray();
var tb3 = x.Select(xx => P3LS0tempSplines[2].Interpolate(xx)).ToArray();

var ta2 = x.Select(xx => P2LS0tempSplines[0].Interpolate(xx)).ToArray();
var tb2 = x.Select(xx => P2LS0tempSplines[1].Interpolate(xx)).ToArray();

Gnuplot plt = new Gnuplot();

plt.PlotXY(x, ta3, title : "Temperature A", format: new PlotFormat("-r"));
plt.PlotXY(x, tb3, title : "Temperature B", format: new PlotFormat("-k"));

plt.PlotXY(x, ta2, title : "Temperature A", format: new PlotFormat("--r"));
plt.PlotXY(x, tb2, title : "Temperature B", format: new PlotFormat("--k"));


plt.Cmd("set grid");
plt.PlotNow().Display();

//ExportData($"Temperature3_A_0.txt",x, ta3);
//ExportData($"Temperature3_B_0.txt",x, tb3);

//ExportData($"Temperature2_A_0.txt",x, ta2);
//ExportData($"Temperature2_B_0.txt",x, tb2);


The same consideration but for heat fluxes in interface normal direction.

In [ ]:
var tya3 = x.Select(xx => -P3LS0tempGradYSplines[0].Interpolate(xx)).ToArray();
var tyb3 = x.Select(xx => -P3LS0tempGradYSplines[2].Interpolate(xx)).ToArray();
var tyg = x.Select(xx => -P3LS0tempGradYSplines[0].Interpolate(xx) + P3LS0tempGradYSplines[2].Interpolate(xx)).ToArray();

var tya2 = x.Select(xx => -P2LS0tempGradYSplines[0].Interpolate(xx)).ToArray();
var tyb2 = x.Select(xx => -P2LS0tempGradYSplines[1].Interpolate(xx)).ToArray();
    
Gnuplot plt = new Gnuplot();

plt.PlotXY(x.Select(xx => Math.Abs(xx)), tya3.Select(xx => Math.Abs(xx)), title : "HeatFlux A", format: new PlotFormat("-r"));
plt.PlotXY(x.Select(xx => Math.Abs(xx)), tyb3.Select(xx => Math.Abs(xx)), title : "HeatFlux B", format: new PlotFormat("-k"));
plt.PlotXY(x.Select(xx => Math.Abs(xx)), tyg.Select(xx => Math.Abs(xx)), title : "HeatFlux Total", format: new PlotFormat("-b"));

plt.PlotXY(x.Select(xx => Math.Abs(xx)), tya2.Select(xx => Math.Abs(xx)), title : "HeatFlux A", format: new PlotFormat("--r"));
plt.PlotXY(x.Select(xx => Math.Abs(xx)), tyb2.Select(xx => Math.Abs(xx)), title : "HeatFlux B", format: new PlotFormat("--k"));

plt.Cmd("set grid");
plt.PlotNow().Display();

//ExportData($"HeatFlux3_A_0.txt",x, tya3);
//ExportData($"HeatFlux3_B_0.txt",x, tyb3);

//ExportData($"HeatFlux2_A_0.txt",x, tya2);
//ExportData($"HeatFlux2_B_0.txt",x, tyb2);